In [1]:
import json
from tqdm import tqdm
import logging
from datasets import Dataset
import datetime


/home/dpopov/work/rudeft/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [2]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

Предполагается, что оригинальный датасет лежит в директории `../hf_deft_ru/`

In [3]:
# orig files
dev_file = "../hf_deft_ru/dev.json"
train_file = "../hf_deft_ru/train.json"

А преобразованный датасет сохраняем в директорию `../ls_deft_ru/`

### Конвертер выгружаемого файла Label studio с разметкой (переводом) от разметчиков к HF формату

- выполняется по [задаче](https://tsa04.isa.ru/ibuyanov/rudeft/-/issues/12)


##### Задача
- фильтруем аннотации из файла LS, где нет в списке `results` элемента `choices`
- получаем все измененные переводы, которые надо обновить в датасете
- составляем словарь id2text_rus, где text_rus берется уже из разметки в списке `results`
- проходимся и обновляем по id text_rus

- паралелльно еще просто собираем все id на будущее, для того чтобы знать какие id имеют верифицированный перевод

7. Составить соответствие HF датасета и выкачанного из Label Studio
- состыковать по `["meta"]["id"]`. т.е. внутренний ID HF датасета, который передавали в поле `meta` в LS
- загрузить датасет в список (каждый элемент - одна строка датасета, т.е. словарь)
- создать индекс соответствия `id -> элемент в списке` (т.е. к номеру элемента в списке)
- искать id из LS данных в индексе

In [4]:
def convert_translation_ls2hf(ls_file, hf_file):
    """ Функция конвертации верифицированного перевода LS (Label Studio) в HF (Hugging Face)  формат

    Текст на русском языке обновить в ориг. датасете для тех предложений, где есть choices (т.е. есть метка "Неправильный перевод")
    
    :param ls_file: путь к файлу выгрузки размеченных данных в LS формате
    :param hf_file: путь к файлу датасета в HF формате, к которому необходимо добавить русскую разметку.

    """
    cnt_processed = 0
    cnt_added_in_hf = 0
    cnt_empty_added_in_hf = 0
    cnt_skip = 0
    cnt_not_in_this_hf_file = 0



    # считываем данные из hf датасета   
    hf_data = Dataset.from_json(hf_file)        # list of dicts (dict = one line from hf dataset)
    hf_data = hf_data.to_list()

    
    # создаем индекс для быстрого доступа к нужному элементу HF по ID
    id2hfindex = {}
    for index, elem in tqdm(enumerate(hf_data), total=len(hf_data), desc="Create index id2hfindex"):
        id2hfindex[elem["id"]] = index

    # считываем данные с аннотациями (разметкой)
    with open(ls_file, 'r') as f:
        ls_data = json.load(f)

    
    ids_verified_sent = []      # список id которые имеют верифицированный перевод
        
    for ls_elem in tqdm(ls_data, desc="Processing LS annotations"):
        ids_verified_sent.append(ls_elem['meta']['id'])

        cnt_processed += 1

        annotations = ls_elem.get("annotations", [])
        if len(annotations) != 1:
            logging.warning(f"Len annotations != 1. Check ID: '{ls_elem.get('id', 'no_id')}'.")
        annotation = annotations[0]

        HAVE_CHOISES = False                    # флаг для пропуска аннотации
        for res in annotation.get("result", []):
            if res["type"] == "choices":
                HAVE_CHOISES = True            # флаг устаравливаем в True, если встречается хоть одна "галочка"
                break

        if not HAVE_CHOISES:                         # переходим к следующему элементу LS, если флаг False
            cnt_skip += 1
            logging.info(f"LS. ID: '{ls_elem['meta']['id']}' has not choice `wrong_translation`! Skip!")
            continue

        new_text_rus = ""
        id2new_text_rus = {}

        for res in annotation.get("result", []):
            
            try:
                if  res["type"] == "textarea" and \
                        res['from_name'] == "text_rus" and \
                        res['to_name'] == "text" and \
                        res["origin"] == "manual":
                    # это русский перевод
                    
                    new_text_rus = res["value"]["text"][0]
                    orig_id = ls_elem['meta']['id']
                    id2new_text_rus[orig_id] = new_text_rus     # зафиксировали этот id 
                    
            except Exception as e:
                logging.error(f"LS. ID: '{ls_elem['meta']['id']}' processing and have error!")
                raise(e)
        
        
        # добавление данных в HF датасет
        hf_id = ls_elem["meta"]["id"]

        if hf_id in id2hfindex and new_text_rus:
            logging.info(f"LS. ID: '{ls_elem['meta']['id']}' has changed translation and it is in this HF file! Processing...")

            num_in_index = id2hfindex[hf_id]
            logging.debug(f"Before: {json.dumps({'id': hf_data[num_in_index]['id'], 'text_rus': hf_data[num_in_index]['text_rus'] }, ensure_ascii=False)}")
            hf_data[num_in_index]["text_rus"] = new_text_rus
            logging.debug(f" After: {json.dumps({'id': hf_data[num_in_index]['id'], 'text_rus': hf_data[num_in_index]['text_rus'] }, ensure_ascii=False)}")

            cnt_added_in_hf += 1
        else:
            # в текущем HF файле нет предложения из LS 
            # значит предложение из другого HF файла
            cnt_not_in_this_hf_file += 1
            logging.info(f"LS. ID: '{ls_elem['meta']['id']}' has changed translation and it is NOT in this HF file! Processing...")

    
    now = datetime.datetime.now()
    formatted_date = now.strftime("%d_%m_%Y_%H_%M")
    with open(f"./ids_verified_sent_{formatted_date}.json", "w") as f:
        json.dump(ids_verified_sent, f, indent=4)

    logging.info(f"Processed: {cnt_processed}, \tAdded: {cnt_added_in_hf}, \tAdded_empty: {cnt_empty_added_in_hf}, \tSent_not_in_this_HF: {cnt_not_in_this_hf_file}, \tSkip_total: {cnt_skip}, \tLen_hf_file: {len(hf_data)}.")
    
    # Записываем данные в HF датасет
    dataset = Dataset.from_dict({key: [d[key] for d in hf_data] for key in hf_data[0]})
    dataset.to_json(hf_file)



In [5]:
ls_annotations = "../ls_annotations/project-7-at-2024-10-07-08-39-1f7a1eeb.json"

In [6]:
dev_file_hf = "../hf_deft_ru/dev.json"
train_file_hf = "../hf_deft_ru/train.json"

In [7]:
convert_translation_ls2hf(ls_file=ls_annotations, hf_file=dev_file_hf)

Generating train split: 1179 examples [00:00, 37038.78 examples/s]


Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 35.20ba/s]


In [8]:
convert_translation_ls2hf(ls_file=ls_annotations, hf_file=train_file_hf)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 24184 examples [00:00, 149013.38 examples/s]
Creating json from Arrow format: 100%|██████████| 25/25 [00:00<00:00, 35.35ba/s]


In [9]:
!grep "Processed:" log.log 

2024-10-07 17:08:12,852 INFO [608458]: root: Processed: 200, 	Added: 6, 	Added_empty: 0, 	Sent_not_in_this_HF: 22, 	Skip_total: 172, 	Len_hf_file: 1179.
2024-10-07 17:08:39,945 INFO [608458]: root: Processed: 200, 	Added: 22, 	Added_empty: 0, 	Sent_not_in_this_HF: 6, 	Skip_total: 172, 	Len_hf_file: 24184.
